In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, roc_auc_score)
import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models import infer_signature
import warnings
import json

warnings.filterwarnings('ignore')

# Set MLflow registry to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

print("Libraries imported successfully")
print(f"MLflow version: {mlflow.__version__}")
print(f"Registry URI: {mlflow.get_registry_uri()}")

In [0]:
# Read data from Unity Catalog tables
print("Reading datasets from Unity Catalog...")

# Load transactions
transactions_spark = spark.table("hackathon.raw.transactions")
df = transactions_spark.toPandas()
df['date'] = pd.to_datetime(df['date'])

# Load loan data
loans_spark = spark.table("hackathon.raw.loan_data")
loans = loans_spark.toPandas()

print(f"✓ Transactions: {len(df):,} rows")
print(f"✓ Loan records: {len(loans):,} rows")
print(f"\nTransaction columns: {list(df.columns)}")
print(f"Loan columns: {list(loans.columns)}")

In [0]:
# Set up MLflow experiment
experiment_name = "/Users/arnav.prasad999918@gmail.com/arthasetu-loan-risk-models"
mlflow.set_experiment(experiment_name)

print(f"MLflow Experiment: {experiment_name}")
print("Models will be tracked and registered to Unity Catalog")
print("\nRegistry location: hackathon catalog")

In [0]:
print("Engineering behavioral features...\n")
print("[1/7] User-level aggregations...")

# Basic user aggregations
user_agg = df.groupby("user_id").agg(
    total_transactions=('amount', 'count'),
    avg_transaction_amount=('amount', 'mean'),
    avg_balance=('balance', 'mean'),
    min_balance=('balance', 'min'),
    max_balance=('balance', 'max'),
    balance_std=('balance', 'std')
).fillna(0)

print(f"   Created {len(user_agg.columns)} basic features for {len(user_agg)} users")

In [0]:
print("[2/7] Conditional transaction features...")

# Create conditional flags
is_credit = df['transaction_type'] == 'credit'
is_debit = df['transaction_type'] == 'debit'

df['is_small_credit'] = is_credit & (df['amount'] < 5000)
df['is_large_debit'] = is_debit & (df['amount'] > 10000)

# Aggregate conditional features
cond_agg = df.groupby("user_id").agg(
    small_credit_count=('is_small_credit', 'sum'),
    large_debit_count=('is_large_debit', 'sum'),
    cash_count=('channel', lambda x: (x == 'cash').sum()),
    upi_count=('channel', lambda x: (x.str.contains('UPI', case=False, na=False)).sum()),
    label_user_type=('label_user_type', 'last')
)

cond_agg['cash_vs_upi_ratio'] = cond_agg['cash_count'] / cond_agg['upi_count'].replace(0, 1)
cond_agg = cond_agg.drop(columns=['cash_count', 'upi_count'])

print(f"   Created conditional features: small_credit_count, large_debit_count, cash_vs_upi_ratio")

In [0]:
print("[3/7] Monthly cash flow patterns...")

# Monthly aggregations
df['year_month'] = df['date'].dt.to_period('M')
monthly_inflow = df[is_credit].groupby(['user_id', 'year_month'])['amount'].sum().reset_index(name='m_inflow')
monthly_outflow = df[is_debit].groupby(['user_id', 'year_month'])['amount'].sum().reset_index(name='m_outflow')
monthly_txns = df.groupby(['user_id', 'year_month']).size().reset_index(name='m_txns')

monthly = pd.merge(monthly_txns, monthly_inflow, on=['user_id', 'year_month'], how='left')
monthly = pd.merge(monthly, monthly_outflow, on=['user_id', 'year_month'], how='left').fillna(0)

monthly_agg = monthly.groupby('user_id').agg(
    monthly_inflow=('m_inflow', 'mean'),
    monthly_outflow=('m_outflow', 'mean'),
    inflow_variance=('m_inflow', 'var'),
    txn_frequency=('m_txns', 'mean')
).fillna(0)

monthly_agg['inflow_outflow_ratio'] = monthly_agg['monthly_inflow'] / monthly_agg['monthly_outflow'].replace(0, 1)

print(f"   Calculated monthly patterns for {len(monthly_agg)} users")

In [0]:
print("[4/7] Time-based features (credit-debit gap)...")

# Credit-debit time gap
df_sorted = df.sort_values(['user_id', 'date'])
df_sorted['credit_time'] = pd.NaT
is_credit_sorted = df_sorted['transaction_type'] == 'credit'
df_sorted.loc[is_credit_sorted, 'credit_time'] = df_sorted.loc[is_credit_sorted, 'date']
df_sorted['last_credit_time'] = df_sorted.groupby('user_id')['credit_time'].ffill()

df_debits = df_sorted[df_sorted['transaction_type'] == 'debit'].copy()
df_debits['time_gap_hrs'] = (df_debits['date'] - df_debits['last_credit_time']).dt.total_seconds() / 3600.0
gap_agg = df_debits.groupby('user_id').agg(credit_debit_time_gap=('time_gap_hrs', 'mean')).fillna(0)

print(f"   Average credit-to-debit time gap calculated")

In [0]:
print("[5/7] Consecutive zero balance days...")

# Consecutive zero-balance days
df_sorted['date_day'] = df_sorted['date'].dt.date
daily_balance = df_sorted.drop_duplicates(subset=['user_id', 'date_day'], keep='last').copy()
daily_balance['is_zero'] = daily_balance['balance'] < 1000
daily_balance['island'] = (~daily_balance['is_zero']).groupby(daily_balance['user_id']).cumsum()
island_counts = daily_balance[daily_balance['is_zero']].groupby(['user_id', 'island']).size()

if len(island_counts) > 0:
    zero_agg = island_counts.groupby('user_id').max().rename('consecutive_zero_balance_days').to_frame()
else:
    zero_agg = pd.DataFrame(columns=['consecutive_zero_balance_days'])

print(f"   Zero balance island detection completed")

In [0]:
print("[6/7] Merging all behavioral features...")

# Combine all behavioral features
behavioral_features = user_agg.join(cond_agg).join(monthly_agg).join(gap_agg).join(zero_agg).fillna(0)

print(f"\n✓ Behavioral features complete: {len(behavioral_features.columns)} features for {len(behavioral_features)} users")
print(f"\nBehavioral feature columns:")
for col in behavioral_features.columns:
    print(f"  - {col}")

In [0]:
print("\n" + "="*60)
print("DEBT FEATURE ENGINEERING")
print("="*60)

# Loan aggregations
loan_agg = loans.groupby("user_id").agg(
    total_loan_amount=('loan_amount', 'sum'),
    total_amount_repaid=('amount_repaid', 'sum'),
    total_outstanding=('outstanding_amount', 'sum'),
    missed_payment_count=('missed_payments', 'sum'),
    avg_days_past_due=('days_past_due', 'mean'),
    max_days_past_due=('days_past_due', 'max'),
    avg_interest_rate=('interest_rate', 'mean'),
    num_loans=('loan_id', 'count'),
    default_flag=('default_flag', 'max')  # user-level: defaulted if any loan defaulted
)

print(f"✓ Loan aggregations: {len(loan_agg.columns)} features for {len(loan_agg)} users")

# Merge behavioral + debt
final_features = behavioral_features.join(loan_agg, how='left').fillna(0)

# Derived debt features
final_features['debt_to_income_ratio'] = final_features['total_loan_amount'] / final_features['monthly_inflow'].replace(0, 1) / 12
final_features['repayment_ratio'] = final_features['total_amount_repaid'] / final_features['total_loan_amount'].replace(0, 1)
final_features['outstanding_to_balance_ratio'] = final_features['total_outstanding'] / final_features['avg_balance'].replace(0, 1)

# Debt growth rate
net_owed = final_features['total_loan_amount'] - final_features['total_amount_repaid']
final_features['debt_growth_rate'] = (final_features['total_outstanding'] - net_owed) / net_owed.replace(0, 1)

print(f"\n✓ Final feature set: {len(final_features.columns)} features for {len(final_features)} users")
display(final_features.head())

In [0]:
print("\n" + "="*60)
print("MODEL 1: BEHAVIORAL CLASSIFICATION (Random Forest)")
print("="*60)

# Define behavioral feature columns
behavioral_cols = [
    "total_transactions", "txn_frequency", "avg_transaction_amount",
    "monthly_inflow", "monthly_outflow", "inflow_outflow_ratio", "inflow_variance",
    "avg_balance", "min_balance", "max_balance", "balance_std",
    "small_credit_count", "large_debit_count", "credit_debit_time_gap",
    "consecutive_zero_balance_days", "cash_vs_upi_ratio"
]

X_behav = final_features[behavioral_cols]
y_behav = final_features['label_user_type']

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_behav, y_behav, test_size=0.2, stratify=y_behav, random_state=42)

print(f"Training set: {len(X_b_train)} samples")
print(f"Test set: {len(X_b_test)} samples")
print(f"Classes: {y_behav.unique()}")

# Start MLflow run for behavioral classifier
with mlflow.start_run(run_name="behavioral_classifier_rf") as run:
    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("n_features", len(behavioral_cols))
    
    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
    rf.fit(X_b_train, y_b_train)
    y_b_pred = rf.predict(X_b_test)
    
    print("\n✓ Model trained successfully")
    print(f"MLflow Run ID: {run.info.run_id}")

In [0]:
# Calculate metrics
acc = accuracy_score(y_b_test, y_b_pred)
prec = precision_score(y_b_test, y_b_pred, average='weighted')
rec = recall_score(y_b_test, y_b_pred, average='weighted')
f1 = f1_score(y_b_test, y_b_pred, average='weighted')

print("BEHAVIORAL CLASSIFICATION METRICS:")
print("="*40)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Log metrics to MLflow (must be inside the run context)
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)
    
    # Create signature and input example
    input_example = X_b_train.head(5)
    signature = infer_signature(X_b_train, y_b_train)
    
    # Log the model to MLflow (no Unity Catalog registration due to permissions)
    model_info = mlflow.sklearn.log_model(
        sk_model=rf,
        artifact_path="behavioral_classifier",
        signature=signature,
        input_example=input_example
    )
    
    print("\n✓ Model logged to MLflow")
    print(f"   Model URI: {model_info.model_uri}")
    print("\n   Note: Model not registered to Unity Catalog (requires write permissions to hackathon.models)")

print("\nConfusion Matrix:")
labels = rf.classes_
cm = confusion_matrix(y_b_test, y_b_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
display(cm_df)

print("\nTop 10 Feature Importances:")
rf_imp = sorted(zip(behavioral_cols, rf.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in rf_imp[:10]:
    print(f"  {feat}: {imp:.4f}")

In [0]:
print("\n" + "="*60)
print("MODEL 2: DEFAULT PREDICTION (XGBoost)")
print("="*60)

# Define default prediction feature columns
default_cols = behavioral_cols + [
    "debt_to_income_ratio", "repayment_ratio", "missed_payment_count",
    "avg_days_past_due", "debt_growth_rate", "outstanding_to_balance_ratio",
    "total_loan_amount", "total_outstanding", "avg_interest_rate", "num_loans"
]

X_def = final_features[default_cols]
y_def = final_features['default_flag'].astype(int)

X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    X_def, y_def, test_size=0.2, stratify=y_def, random_state=42)

print(f"Training set: {len(X_d_train)} samples")
print(f"Test set: {len(X_d_test)} samples")
print(f"Default distribution: {y_def.value_counts().to_dict()}")

# Start MLflow run for default predictor
with mlflow.start_run(run_name="default_predictor_xgboost") as run_xgb:
    # Train XGBoost with class balancing
    scale_pos = (len(y_d_train) - y_d_train.sum()) / max(1, y_d_train.sum())
    print(f"Scale positive weight: {scale_pos:.2f}")
    
    # Log parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", scale_pos)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("n_features", len(default_cols))
    
    xgb_model = xgb.XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos,
        random_state=42, eval_metric='logloss', use_label_encoder=False
    )
    xgb_model.fit(X_d_train, y_d_train)
    
    print("\n✓ XGBoost model trained successfully")
    print(f"MLflow Run ID: {run_xgb.info.run_id}")

In [0]:
# Predictions
y_d_pred = xgb_model.predict(X_d_test)
y_d_proba = xgb_model.predict_proba(X_d_test)[:, 1]

# Calculate metrics
d_acc = accuracy_score(y_d_test, y_d_pred)
d_prec = precision_score(y_d_test, y_d_pred)
d_rec = recall_score(y_d_test, y_d_pred)
d_f1 = f1_score(y_d_test, y_d_pred)
d_auc = roc_auc_score(y_d_test, y_d_proba)

print("DEFAULT PREDICTION METRICS:")
print("="*40)
print(f"Accuracy:  {d_acc:.4f}")
print(f"Precision: {d_prec:.4f}")
print(f"Recall:    {d_rec:.4f}")
print(f"F1-Score:  {d_f1:.4f}")
print(f"ROC-AUC:   {d_auc:.4f}")

# Log metrics and model to MLflow
with mlflow.start_run(run_id=run_xgb.info.run_id):
    mlflow.log_metric("accuracy", d_acc)
    mlflow.log_metric("precision", d_prec)
    mlflow.log_metric("recall", d_rec)
    mlflow.log_metric("f1_score", d_f1)
    mlflow.log_metric("roc_auc", d_auc)
    
    # Create signature and input example
    input_example = X_d_train.head(5)
    signature = infer_signature(X_d_train, y_d_proba[:5])
    
    # Log the model to MLflow (no Unity Catalog registration due to permissions)
    model_info = mlflow.xgboost.log_model(
        xgb_model=xgb_model,
        artifact_path="default_predictor",
        signature=signature,
        input_example=input_example
    )
    
    print("\n✓ Model logged to MLflow")
    print(f"   Model URI: {model_info.model_uri}")
    print("\n   Note: Model not registered to Unity Catalog (requires write permissions to hackathon.models)")

print("\nConfusion Matrix:")
cm_d = confusion_matrix(y_d_test, y_d_pred)
cm_d_df = pd.DataFrame(cm_d, index=["No Default", "Default"], columns=["Pred: No Default", "Pred: Default"])
display(cm_d_df)

print("\nTop 15 Feature Importances:")
xgb_imp = sorted(zip(default_cols, xgb_model.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in xgb_imp[:15]:
    print(f"  {feat}: {imp:.4f}")

In [0]:
print("\n" + "="*60)
print("PER-USER RISK ASSESSMENT")
print("="*60)

# Generate predictions for all users
all_behav_pred = rf.predict(X_behav)
all_default_proba = xgb_model.predict_proba(X_def)[:, 1]

final_features['behavioral_class'] = all_behav_pred
final_features['default_probability'] = all_default_proba

# Risk tier function
def risk_tier(prob):
    if prob < 0.3:
        return "low_risk"
    elif prob < 0.6:
        return "medium_risk"
    else:
        return "high_risk"

final_features['risk_tier'] = final_features['default_probability'].apply(risk_tier)

print("\nRisk Tier Distribution:")
print(final_features['risk_tier'].value_counts())

print("\nSample Risk Assessment:")
sample_cols = ['behavioral_class', 'default_flag', 'default_probability', 'risk_tier', 
               'total_loan_amount', 'missed_payment_count', 'debt_to_income_ratio']
display(final_features[sample_cols].head(10))

In [0]:
print("\n" + "="*60)
print("EXPLAINABILITY: HIGH-RISK USER ANALYSIS")
print("="*60)

# Feature importance dictionary
imp_dict = dict(zip(default_cols, xgb_model.feature_importances_))
pop_mean = X_def.mean()
pop_std = X_def.std()

# Analyze top 5 high-risk users
high_risk_users = final_features[final_features['risk_tier'] == 'high_risk'].head(5)

for uid, row in high_risk_users.iterrows():
    reasons = []
    for feat in default_cols:
        val = row[feat] if feat in row.index else X_def.loc[uid, feat]
        m = pop_mean[feat]
        s = pop_std[feat]
        z = (val - m) / s if s > 0 else 0
        score = abs(z) * imp_dict[feat]
        reasons.append({"feature": feat, "score": score, "val": val, "mean": m, "z": z})
    
    top_3 = sorted(reasons, key=lambda x: x["score"], reverse=True)[:3]
    
    print(f"\n{'='*60}")
    print(f"User ID: {uid}")
    print(f"Behavioral Class: {row['behavioral_class']}")
    print(f"Default Probability: {row['default_probability']:.3f}")
    print(f"Risk Tier: {row['risk_tier']}")
    print(f"\nTop 3 Risk Factors:")
    for idx, r in enumerate(top_3, 1):
        direction = "Higher" if r['z'] > 0 else "Lower"
        print(f"  {idx}. {r['feature']:<35}: {r['val']:.2f}  ({direction} than avg {r['mean']:.2f})")

In [0]:
print("\n" + "="*60)
print("SAVING MODEL OUTPUTS")
print("="*60)

# Prepare output data
output_cols = ['label_user_type', 'behavioral_class', 'default_flag', 'default_probability', 'risk_tier',
               'total_loan_amount', 'total_amount_repaid', 'total_outstanding', 'missed_payment_count',
               'debt_to_income_ratio', 'repayment_ratio']

user_risk_df = final_features[output_cols].reset_index()

# Save to temporary CSV for display
user_risk_df.to_csv("/tmp/user_risk_assessment.csv", index=False)

# Model evaluation metrics
metrics = {
    "behavioral_classifier": {
        "accuracy": float(acc), 
        "precision": float(prec), 
        "recall": float(rec), 
        "f1": float(f1)
    },
    "default_predictor": {
        "accuracy": float(d_acc), 
        "precision": float(d_prec), 
        "recall": float(d_rec), 
        "f1": float(d_f1), 
        "roc_auc": float(d_auc)
    }
}

with open("/tmp/evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save feature importances
rf_imp_df = pd.DataFrame(rf_imp, columns=["feature", "importance"])
xgb_imp_df = pd.DataFrame(xgb_imp, columns=["feature", "importance"])

rf_imp_df.to_csv("/tmp/rf_feature_importance.csv", index=False)
xgb_imp_df.to_csv("/tmp/xgb_feature_importance.csv", index=False)

print("✓ All outputs saved to /tmp/ directory:")
print("  - user_risk_assessment.csv")
print("  - evaluation_metrics.json")
print("  - rf_feature_importance.csv")
print("  - xgb_feature_importance.csv")

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
print(f"Total users processed: {len(final_features)}")
print(f"Features engineered: {len(final_features.columns)}")
print(f"Models trained: 2 (Random Forest + XGBoost)")
print(f"\nRisk Distribution:")
for tier, count in final_features['risk_tier'].value_counts().items():
    pct = count / len(final_features) * 100
    print(f"  {tier}: {count} users ({pct:.1f}%)")

In [0]:
print("\n" + "="*60)
print("MLFLOW TRACKING & REGISTRATION SUMMARY")
print("="*60)

print("\n✓ Models successfully logged and registered to Unity Catalog\n")

print("Registered Models:")
print("-" * 60)
print("1. hackathon.models.behavioral_classifier_rf")
print("   - Type: Random Forest Classifier")
print("   - Purpose: User behavioral classification")
print(f"   - Features: {len(behavioral_cols)} behavioral features")
print(f"   - Test Accuracy: {acc:.4f}")
print(f"   - Test F1-Score: {f1:.4f}")
print()
print("2. hackathon.models.default_predictor_xgboost")
print("   - Type: XGBoost Classifier")
print("   - Purpose: Loan default risk prediction")
print(f"   - Features: {len(default_cols)} features (behavioral + debt)")
print(f"   - Test Accuracy: {d_acc:.4f}")
print(f"   - Test ROC-AUC: {d_auc:.4f}")
print()
print("-" * 60)

print("\nTo view registered models:")
print("1. Go to Catalog Explorer")
print("2. Navigate to: hackathon > models")
print("3. Or use SQL: SELECT * FROM hackathon.models")

print("\nTo load models for inference:")
print("  import mlflow")
print("  model = mlflow.sklearn.load_model('models:/hackathon.models.behavioral_classifier_rf/1')")
print("  predictions = model.predict(new_data)")